# SymboLM 200-Question Mixed Benchmark Evaluation

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Exoskeleton:** Antigravity (Google DeepMind)  
**Hardware:** Kaggle NVIDIA Tesla T4 GPU  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Scope of Evaluation (200 Questions):
- **130 Competition & Invariant Mathematics:** Polynomial invariants, CRT modular systems, modular exponentiation, Legendre prime valuations, combinatorics/derangements, inradius geometry, Diophantine harmonic rates.
- **45 General Knowledge & Factual QA:** Testing zero-shot factual recall routing (`<reg:bypass>`) without reasoning bloat.
- **25 Communicative Intent Dialogue:** Testing communicative alignment scratchpads (`<reg:intent>`).

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Install required libraries
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft accelerate sympy datasets huggingface_hub


In [ ]:
# Cell 3: Clone latest SymboLM repository
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
print('[OK] Successfully cloned latest SymboLM repository!')


In [ ]:
# Cell 4: Mount Stage 3.5 Concept GRPO Policy Adapter
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/stage3_concept_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for Stage 3 adapter in mounted kernel sources
for p in Path('/kaggle/input').glob('**/stage3_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found Stage 3 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. Search for stage 3 tarball archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*stage3*policy*.tar.gz'):
        print(f'[OK] Found Stage 3 tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

# 3. Fallback: Download from Hugging Face Hub
if not found or not (target_dir / 'adapter_model.safetensors').exists():
    print('[INFO] Pulling Stage 3.5 adapter directly from Hugging Face Hub...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='gyawalisanish0/symboLM-checkpoints',
        allow_patterns='stage3_concept_adapter/*',
        local_dir='/kaggle/working/hf_download'
    )
    src_hf = Path('/kaggle/working/hf_download/stage3_concept_adapter')
    for item in src_hf.iterdir():
        if item.is_file():
            shutil.copy2(item, target_dir / item.name)

print('[Status] Validating Adapter Files:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] Stage 3.5 Concept GRPO Policy Adapter is ready for evaluation!')


In [ ]:
# Cell 5: Run the 200-Question Mixed Benchmark Audit
!python /kaggle/working/symboLM/eval/run_200_mixed_benchmark.py \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --adapter_path /kaggle/working/symboLM/checkpoints/stage3_concept_adapter \
  --data_path /kaggle/working/symboLM/data/eval_200_mixed_benchmark.jsonl \
  --output_path /kaggle/working/eval_200_results.json \
  --report_path /kaggle/working/EVAL_200_BENCHMARK_REPORT.md \
  --max_tokens 256 \
  --temperature 0.2


In [ ]:
# Cell 6: Display Markdown Summary Report
import os
rep_path = '/kaggle/working/EVAL_200_BENCHMARK_REPORT.md'
if os.path.exists(rep_path):
    with open(rep_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('[Warning] Report not found at', rep_path)
print('[COMPLETE] 200-Question Mixed Benchmark execution finished!')
